In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import numpy as np

import channel_design
from channel_design import (
    annualised_cost,
    candidate_designs,
    route_with_head_loss_iteration,
    start_map,
)

In [3]:
#plain the model scans
# m = start_map()
# m

In [8]:

# ---- Paths ----
PROJECT_DIR = 'run1'
ROUTES_DIR = os.path.join(PROJECT_DIR, '1_Routes')
SEQUENTIAL_RUN_TEMPLATE = os.path.join('inputs/Sequential_Run.csv')

DEM_PATH = "inputs/DEM/merged_dem.tif"
ROCK_DEPTH_PATH = "inputs/DEM/low_res_resampled.tif"

# ---- Routing area (lon/lat): left, bottom, right, top ----
ROUTING_BOUNDS = (147, -21, 149, -19)

# ---- Channel hydraulics ----
MANNING_N = 0.03
SIDE_SLOPE = 2  # horizontal : vertical

# ---- Router settings ----
SAMPLE_F = 250
MAX_FILL_SIPHON_ENDS = 5

# ---- Sequential run settings ----
MAX_RUNS = 4
RESIDUAL_HEAD_LOSS_TOL_M = 0.1

# ---- Per-section inputs ----
SECTIONS = {
    'section1': dict(
        start=(147.31, -19.81),
        finish=(147.857, -19.978),       # before pumping station
        start_elevation=36,              # DFL
        gross_diversions_m3_s=19.4,
        default_linear_cost=583930,
        total_loss_GL=8.7,
        approximate_length_km=67,        # used for the length penalty
        no_cells=1991,
    ),
    'section2': dict(
        start=(147.866, -19.974),        # after pumping station
        finish=(148.079, -20.086),       # before Euri Creek
        start_elevation=40,              # DFL
        gross_diversions_m3_s=10,
        default_linear_cost=577045,
        total_loss_GL=2.9,
        approximate_length_km=40,
        no_cells=805,
    ),
}

# Pass the settings above to channel_design
channel_design.configure(
    ROUTES_DIR=ROUTES_DIR,
    SEQUENTIAL_RUN_TEMPLATE=SEQUENTIAL_RUN_TEMPLATE,
    DEM_PATH=DEM_PATH,
    ROCK_DEPTH_PATH=ROCK_DEPTH_PATH,
    ROUTING_BOUNDS=ROUTING_BOUNDS,
    MANNING_N=MANNING_N,
    SIDE_SLOPE=SIDE_SLOPE,
    SAMPLE_F=SAMPLE_F,
    MAX_FILL_SIPHON_ENDS=MAX_FILL_SIPHON_ENDS,
    MAX_RUNS=MAX_RUNS,
    RESIDUAL_HEAD_LOSS_TOL_M=RESIDUAL_HEAD_LOSS_TOL_M,
    SECTIONS=SECTIONS,
)

# Flow velocities (m/s) swept when generating candidate channel designs
VELOCITY_RANGES = {
    'section1': np.arange(0.490, 0.491, 0.01),
    'section2': np.arange(0.425, 0.444, 0.001),
}


version = 'v22'
section = 'section1'

designs = candidate_designs(section, ratios=[5], velocities=VELOCITY_RANGES[section])
[design[5] for design in designs]  # velocities kept

[np.float64(0.49)]

In [10]:
for section, ratio, depth, width, slope_1_in_x, velocity in designs:
    run_name = f'{version}-{section}-r{ratio}-d{depth}-w{width}-S{slope_1_in_x}-v{velocity:.3f}'
    work_dir = os.path.join(ROUTES_DIR, run_name)
    if os.path.isdir(work_dir):
        print("Exists:", work_dir)
        continue
    print(work_dir)
    route_with_head_loss_iteration(number_of_runs=3, width_to_depth_ratio=ratio, water_depth=depth, bed_width=width,
                                   run_name=run_name, section=section, slope_for_channel_dimension=1 / slope_1_in_x)

run1\1_Routes\v22-section1-r5-d2.38-w11.89-S9814-v0.490
Calculated channel dimensions: slope = -0.00010189525168127165, ratio_width_depth = 5, water depth = 2.38 m, bed width = 11.89 m


KeyboardInterrupt: 

In [ ]:
# ---- Archived runs (switched off) ----
# These predate the water_depth / bed_width arguments: route_with_head_loss_iteration now needs them
# (run_channel formats them with :.2f), so add them before switching this on.
RUN_ARCHIVED = False

ARCHIVED_RUNS = [
    # v1: section 1, width:depth 5, fixed slope per velocity
    dict(run_name='v1-reach1-ratio5-velocity0.3',  section='section1', slope_for_channel_dimension=4.28E-05),
    dict(run_name='v1-reach1-ratio5-velocity0.35', section='section1', slope_for_channel_dimension=6.46E-05),
    dict(run_name='v1-reach1-ratio5-velocity0.4',  section='section1', slope_for_channel_dimension=9.22E-05),
    dict(run_name='v1-reach1-ratio5-velocity0.45', section='section1', slope_for_channel_dimension=1.263E-04),
    dict(run_name='v1-reach1-ratio5-velocity0.5',  section='section1', slope_for_channel_dimension=1.672E-04),
    dict(run_name='v1-reach1-ratio5-velocity0.55', section='section1', slope_for_channel_dimension=2.1E-04),
    
]

if RUN_ARCHIVED:
    for kwargs in ARCHIVED_RUNS:
        route_with_head_loss_iteration(number_of_runs=3, width_to_depth_ratio=5, **kwargs)

In [ ]:
# Annualised cost examples
print(annualised_cost(577045))
print(annualised_cost(5000))